# Ham ya da Spam?

🎯 Bu görevin amacı, e-postaları **spam (1)** veya **normal e-posta (0)** olarak sınıflandırmaktır.

🧹 İlk olarak, bu metin verilerine **temizleme (cleaning)** teknikleri uygulanacaktır.

👩🏻‍🔬 Ardından, temizlenmiş metinler **sayısal bir gösterime** dönüştürülecektir.

✉️ Son olarak, her bir e-postayı spam mı yoksa normal mi olduğunu sınıflandırmak için  
***Multinomial Naive Bayes*** modeli uygulanacaktır.


## (0) NTLK kütüphanesi (Doğal Dil Araç Seti)

In [1]:
!pip install nltk

  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 1.2 MB/s  0:00:01 eta 0:00:01
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [nltk]4/5 [nltk]

[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [2]:
# nltk'yi ilk kez içe aktarırken, birkaç yerleşik kütüphaneyi de indirmemiz gerekir.

import nltk

nltk.download('stopwords')
nltk.download('punkt')      # nltk<3.9.0 için
nltk.download('punkt_tab')  # nltk>=3.9.0 için
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/serenkayran/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/serenkayran/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/serenkayran/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/serenkayran/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/serenkayran/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/serenkayran/nltk_data...


True

In [3]:
import pandas as pd

df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/ham_spam_emails.csv")
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


## (1) (Metin) veri setinin temizlenmesi

Veri kümesi, ham [0] veya spam [1] olarak sınıflandırılan e-postalardan oluşur. Tahmin modelini eğitmeden önce veri kümesini temizlemeniz gerekir.

### (1.1) Noktalama İşaretlerini Kaldır

❓ Noktalama işaretlerini kaldırmak için bir işlev oluşturun. Bunu `text` sütununa uygulayın ve çıktıyı `clean_text` adlı veri çerçevesinin yeni bir sütununa ekleyin. ❓

In [5]:
import string

def remove_punctuaction(text):
    return text.translate(str.maketrans('', '', string.punctuation))

In [7]:
remove_punctuaction("Hello !!! How are you")
df["clean_text"] = df["text"].apply(remove_punctuaction)
df[["text" , "clean_text"]].head()

,text,clean_text
0,Subject: naturally irresistible your corporate...,Subject naturally irresistible your corporate ...
1,Subject: the stock trading gunslinger fanny i...,Subject the stock trading gunslinger fanny is...
2,Subject: unbelievable new homes made easy im ...,Subject unbelievable new homes made easy im w...
3,Subject: 4 color printing special request add...,Subject 4 color printing special request addi...
4,"Subject: do not have money , get software cds ...",Subject do not have money get software cds fr...


### (1.2) Küçük Harf

❓ Metni küçük harfe çeviren bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [15]:
df["clean_text"]=[t.lower() for t in df["clean_text"]]

df["clean_text"].head()

0    subject naturally irresistible your corporate ...
1    subject the stock trading gunslinger  fanny is...
2    subject unbelievable new homes made easy  im w...
3    subject  color printing special  request addit...
4    subject do not have money  get software cds fr...
Name: clean_text, dtype: str

### (1.3) Sayıları Kaldır

❓ Metinden sayıları kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın ❓

In [16]:
import re
def remove_numbers(text):
    return re.sub(r"\d+","",text)

    
df["clean_text"] = [remove_numbers(t) for t in df["clean_text"]]
print(df["clean_text"])


0       subject naturally irresistible your corporate ...
1       subject the stock trading gunslinger  fanny is...
2       subject unbelievable new homes made easy  im w...
3       subject  color printing special  request addit...
4       subject do not have money  get software cds fr...
                              ...                        
5723    subject re  research and development charges t...
5724    subject re  receipts from visit  jim   thanks ...
5725    subject re  enron case study update  wow  all ...
5726    subject re  interest  david   please  call shi...
5727    subject news  aurora    update  aurora version...
Name: clean_text, Length: 5728, dtype: str


### (1.4) StopWords'ü kaldırın

❓ Metinden durdurma kelimelerini kaldırmak için bir işlev oluşturun. Bunu `clean_text`'e uygulayın. ❓

In [17]:
from nltk.corpus import stopwords
english_stopwords = set(stopwords.words("english"))
print(english_stopwords)


{'this', 'didn', 'there', "she'll", 'own', 'when', 'having', 'himself', 'shouldn', 'yours', "they'd", 'shan', 'here', 'of', 'hers', 'herself', 'itself', 'had', 'doesn', 'only', 're', "hasn't", 'o', 'down', 't', "he'll", 'm', 'both', 'theirs', "weren't", 'you', 'they', 'not', 'don', 's', 'after', 'them', 'some', 'ourselves', 'been', 'as', 've', 'but', 'couldn', 'below', 'mightn', "you're", 'y', 'we', 'will', 'each', "we're", "i've", "i'm", 'an', 'how', 'his', "don't", 'i', 'which', 'or', 'yourselves', "i'll", 'wasn', 'haven', 'just', 'did', 'if', 'such', "it'd", 'being', 'to', 'it', 'myself', 'needn', 'until', 'at', 'll', 'weren', "they've", 'and', 'too', 'its', 'ours', "you've", 'who', 'are', 'very', 'doing', 'yourself', 'your', 'themselves', 'do', "needn't", 'about', 'am', "we'll", 'through', 'won', 'mustn', 'no', 'can', 'now', "you'll", "aren't", 'above', 'does', 'few', 'further', "should've", "that'll", 'our', "shouldn't", 'again', 'against', 'him', 'because', "wouldn't", 'aren', 'd

In [21]:
def remove_stopwords(text):
    words = text.split()
    clean_words =[word for word in words if word not in english_stopwords]
    return " ".join(clean_words)
df["clean_text"] = [
    remove_stopwords(t) for t in df["clean_text"]
]
df["clean_text"].tail()


5723    subject research development charges gpg forwa...
5724    subject receipts visit jim thanks invitation v...
5725    subject enron case study update wow day super ...
5726    subject interest david please call shirley cre...
5727    subject news aurora update aurora version fast...
Name: clean_text, dtype: str

### (1.5) Lemmatize

❓ Metni lemmatize etmek için bir fonksiyon oluşturun. Çıktının bir kelime listesi değil, tek bir dize olduğundan emin olun. Bunu `clean_text`'e uygulayın. ❓

In [23]:
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

In [28]:
def lemmatize_text(text):
    words = text.split()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)

lemmatize_text("cats dogs studies")

df["clean_text"] = [
    lemmatize_text(t) for t in df["clean_text"]
]

df["clean_text"].head(10)

0    subject naturally irresistible corporate ident...
1    subject stock trading gunslinger fanny merrill...
2    subject unbelievable new home made easy im wan...
3    subject color printing special request additio...
4    subject money get software cd software compati...
5    subject great nnews hello welcome medzonline s...
6    subject hot play motion homeland security inve...
7    subject save money buy getting thing tried cia...
8    subject undeliverable home based business grow...
9    subject save money buy getting thing tried cia...
Name: clean_text, dtype: str

## (2) Bag-of-Words Modellemesi

### (2.1) Metin verilerini sayılara dönüştürme

❓ `clean_text`'i varsayılan CountVectorizer ile Bag-of-Words temsiline vektörleştirin. `X_bow` olarak kaydedin. ❓

In [29]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()

X_bow = vectorizer.fit_transform(df["clean_text"])

### (2.2) Çok terimli Naive Bayes Modellemesi

❓ MultinomialNB modelini bag-of-words verileriyle çapraz doğrulayın. Modelin doğruluğunu puanlayın. ❓

In [34]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_predict
from sklearn.model_selection import cross_val_score

X = X_bow
y = df["spam"]

model = MultinomialNB()


In [35]:

scores = cross_val_score(
    model,
    X,
    y,
    cv=5,
    scoring="accuracy"
)
print(scores)
print("Ortalama doğruluk:", scores.mean())


[0.98691099 0.9895288  0.991274   0.98777293 0.99213974]
Ortalama doğruluk: 0.9895252901681946


In [36]:
print("5 Fold Accuracy:", scores)
print("Mean Accuracy:", scores.mean())

5 Fold Accuracy: [0.98691099 0.9895288  0.991274   0.98777293 0.99213974]
Mean Accuracy: 0.9895252901681946


🏁 Tebrikler!

💾 Not defterinizi git add/commit/push yapmayı unutmayın...

🚀 ... ve bir sonraki challenge'a geçin!